# 6.3 · 层次聚类 / Hierarchical Clustering

> **课程定位 / Where this fits**
> K-Means(6.1)要预先定 K, 且只给一个扁平划分。层次聚类**不需预先定 K**: 自底向上不断合并最近的簇, 生成一棵**树状图(dendrogram)**, 想要几个簇就在合适高度"切一刀"。还能揭示嵌套结构(大群里套小群)。
> Hierarchical clustering builds a tree (dendrogram) by merging nearest clusters — no preset K, and reveals nested structure; cut the tree at any height for K clusters.

> 💡 **面试相关 / Interview-relevant**
> - "凝聚 vs 分裂(agglomerative vs divisive)" ★★★★
> - "linkage 准则: single/complete/average/ward 区别" ★★★★★
> - "树状图怎么读 / 怎么定 K" ★★★★
> - "层次聚类复杂度为什么是 O(n²~n³)" ★★★★（不适合大数据）
> - "层次 vs K-Means" ★★★★

---

## 学习目标 / Learning Objectives
1. 凝聚式层次聚类流程 + 树状图。
2. 四种 **linkage** 准则及其几何后果。
3. 从树状图切割定 K。
4. 复杂度与适用边界。

## 目录 / TOC
1. [凝聚聚类与 linkage ⭐](#1)
2. [🛍️ 数据 + 树状图 ⭐](#2)
3. [linkage 准则对比 ⭐](#3)
4. [切树定 K + 对照 K-Means](#4)
5. [小结](#5)


<a id="1"></a>
## 1. 凝聚聚类与 linkage ⭐ / Agglomerative & Linkage

**凝聚式(agglomerative, 自底向上)**: 每点先各自成簇 → 反复**合并距离最近的两个簇** → 直到剩一个簇。全过程记成一棵树(树状图)。(**分裂式 divisive** 反过来自顶向下, 少用。)

关键是"两个**簇**之间的距离"怎么定 —— **linkage 准则**:

| linkage | 簇间距离定义 | 倾向 |
|---|---|---|
| **single(单链)** | 两簇**最近**点对距离 | 链式, 能捕捉细长/非球形, 但易"链桥" |
| **complete(全链)** | 两簇**最远**点对距离 | 紧凑球形簇, 对异常敏感 |
| **average(平均)** | 所有跨簇点对平均距离 | 折中 |
| **ward** | 合并后**簇内方差增量**最小 | 等大球形簇, 最常用(类似 K-Means 目标) |

ward 与 K-Means 精神一致(都最小化簇内方差), 是默认首选。


<a id="2"></a>
## 2. 数据 + 树状图 ⭐ / Data & Dendrogram

复用 6.1 的 **Mall Customers**(同生成器), 在 income/spending 上聚类并画树状图。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.preprocessing import StandardScaler
sns.set_theme(style="whitegrid")

def make_mall(seed=0):
    rng = np.random.default_rng(seed)
    groups = [((55,50),120,(25,60)),((25,80),35,(18,35)),((85,82),40,(28,42)),
              ((85,18),38,(35,60)),((26,18),35,(40,68))]
    rows=[]
    for (inc,spd),n,(amin,amax) in groups:
        rows.append(np.c_[rng.integers(amin,amax,n), rng.normal(inc,8,n).clip(15,140), rng.normal(spd,9,n).clip(1,99)])
    X=np.vstack(rows); rng.shuffle(X)
    return pd.DataFrame(X, columns=["age","income_k","spending"])

mall = make_mall()
print(f"Mall Customers: {mall.shape}")
Xs = StandardScaler().fit_transform(mall[["income_k","spending"]])

Z = linkage(Xs, method="ward")   # 凝聚, ward linkage; Z 是合并记录
fig, ax = plt.subplots(figsize=(11, 4.5))
dendrogram(Z, truncate_mode="lastp", p=30, ax=ax)
ax.axhline(10, color="r", ls="--", label="切割线 → 5 簇")
ax.set_xlabel("样本(截断)"); ax.set_ylabel("合并距离(ward)"); ax.legend()
ax.set_title("树状图: 自底向上合并; 高度=合并代价; 横切一刀决定簇数")
plt.tight_layout(); plt.show()
print("读法: 越晚合并(越高)的两支越不相似; 在大间隙处切, 得到自然簇数")


<a id="3"></a>
## 3. linkage 准则对比 ⭐ / Linkage Comparison

不同 linkage 在同一数据上给出**完全不同**的簇形状。用月牙数据最能看出差异: single 能捕捉弯曲, ward/complete 偏向球形。


In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.datasets import make_moons
Xm, _ = make_moons(300, noise=0.06, random_state=0)

fig, axes = plt.subplots(1, 4, figsize=(15, 3.6))
for ax, lk in zip(axes, ["single","complete","average","ward"]):
    lab = AgglomerativeClustering(n_clusters=2, linkage=lk).fit_predict(Xm)
    ax.scatter(Xm[:,0], Xm[:,1], c=lab, cmap="coolwarm", s=12)
    ax.set_title(f"linkage={lk}"); ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("月牙数据: single 捕捉弯曲簇; complete/ward 偏向球形(切错)")
plt.tight_layout(); plt.show()
print("single: 抓住两条月牙(链式); complete/average/ward: 球形假设→切错")
print("→ 非球形/细长簇用 single; 紧凑等大簇用 ward")


<a id="4"></a>
## 4. 切树定 K + 对照 K-Means / Cutting & vs K-Means

`fcluster` 按距离阈值或簇数切树。下面在 Mall 上切成 5 簇, 和 K-Means(6.1)对比。


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score

hier_labels = fcluster(Z, t=5, criterion="maxclust")        # 切成 5 簇
km_labels = KMeans(5, n_init=10, random_state=0).fit_predict(Xs)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(Xs[:,0], Xs[:,1], c=hier_labels, cmap="tab10", s=18)
axes[0].set_title(f"层次(ward, 切5簇) silhouette={silhouette_score(Xs,hier_labels):.3f}")
axes[1].scatter(Xs[:,0], Xs[:,1], c=km_labels, cmap="tab10", s=18)
axes[1].set_title(f"K-Means(K=5) silhouette={silhouette_score(Xs,km_labels):.3f}")
for a in axes: a.set_xlabel("income (std)"); a.set_ylabel("spending (std)")
plt.tight_layout(); plt.show()
print(f"两者划分高度一致 ARI = {adjusted_rand_score(hier_labels, km_labels):.3f}")
print("球形簇上层次(ward)与 K-Means 结果接近; 层次的优势是不预设K + 树状结构可解释")


<a id="5"></a>
## 5. 小结 / Summary

```
凝聚式: 每点成簇→反复合并最近两簇→树状图; 不预设K, 切树得任意簇数
linkage: single(最近点对, 链式/非球形) complete(最远, 紧凑) average(折中) ward(方差增量, 默认)
树状图: 高度=合并代价; 在大间隙横切定K
复杂度 O(n²)~O(n³) → 不适合大数据(用 K-Means/Mini-batch)
ward 与 K-Means 同精神(最小化簇内方差), 球形簇上结果相近
```

### 💡 面试速查
1. **凝聚(自底向上合并) vs 分裂(自顶向下)**
2. **linkage**: single=最近点对(链式), complete=最远, ward=方差增量(默认/最常用)
3. **树状图**在大间隙处切定 K; 不需预设 K 是核心优势
4. **复杂度 O(n²~n³)** → 大数据不适用
5. **vs K-Means**: 层次不预设K+有层级结构; K-Means 快、可扩展

### 下一节
**6.4 DBSCAN**——前面都假设簇是团状。DBSCAN 基于**密度**, 能找任意形状的簇、自动识别噪声点、且不需预设簇数。
